# Hindi 25M LM - pretraining on Kaggle
Needs: **Accelerator = GPU T4**, **Internet = On**, and the `hindi-tokens` dataset added as input.
To continue an interrupted run, also add the previous version of THIS notebook's output as an input
(the checkpoint is found automatically).

In [ ]:
import subprocess, os
if not os.path.exists('/kaggle/working/repo'):
    subprocess.run(['git','clone','https://github.com/Keerthi-2012/Monolingual-Transformer-LM.git','/kaggle/working/repo'],check=True)
else:
    subprocess.run(['git','-C','/kaggle/working/repo','pull'],check=True)
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],capture_output=True,text=True).stdout)

In [ ]:
import glob, shutil
# 1) token shards: any input folder containing train_000.bin
hits = glob.glob('/kaggle/input/**/train_000.bin', recursive=True)
assert hits, 'Add the hindi-tokens dataset as an input (see README)'
TOKENS = os.path.dirname(hits[0]); print('tokens:', TOKENS, sorted(os.listdir(TOKENS)))
# 2) resume: newest ckpt_latest.pt found in any attached input
OUT = '/kaggle/working/runs/hindi_25m'; os.makedirs(OUT, exist_ok=True)
cks = sorted(glob.glob('/kaggle/input/**/ckpt_latest.pt', recursive=True), key=os.path.getmtime)
if cks and not os.path.exists(f'{OUT}/ckpt_latest.pt'):
    shutil.copy(cks[-1], OUT); print('resuming from', cks[-1])
    for f in ('train_log.jsonl','ckpt_best.pt','config.json'):
        p = os.path.join(os.path.dirname(cks[-1]), f)
        if os.path.exists(p): shutil.copy(p, OUT)
else:
    print('fresh run' if not cks else 'checkpoint already in working dir')

In [ ]:
# Runs until max_steps or ~10.8 h (then saves a checkpoint and exits cleanly). Re-run a new session to continue.
!cd /kaggle/working/repo/hindi/train && python -u train.py --config ../configs/hindi_25m.json     --token-dir {TOKENS} --out-dir {OUT} --time-limit-min 650

In [ ]:
import json, matplotlib.pyplot as plt
h = [json.loads(l) for l in open(f'{OUT}/train_log.jsonl')]
v = [r for r in h if 'val_loss' in r]
plt.figure(figsize=(7,4.5))
plt.plot([r['step'] for r in h], [r['train_loss'] for r in h], alpha=.5, label='train loss')
plt.plot([r['step'] for r in v], [r['val_loss'] for r in v], marker='o', label='validation loss')
plt.title('Hindi 25M LM: loss curves'); plt.xlabel('Optimizer step'); plt.ylabel('Cross-entropy loss'); plt.legend()
plt.savefig(f'{OUT}/loss_curve.png', dpi=150); print('last step', h[-1]['step'], 'last val', v[-1]['val_loss'] if v else None)

## Evaluation (run only after training has reached the final step)
Writes PPL/BPB, generation metrics and attention figures into `/kaggle/working/report/`.

In [ ]:
!pip -q install sacrebleu
!apt-get -qq install -y fonts-noto-core > /dev/null 2>&1; fc-cache -f > /dev/null 2>&1
import os; os.makedirs('/kaggle/working/report/attn', exist_ok=True)
CK = f'{OUT}/ckpt_best.pt'
!cd /kaggle/working/repo/hindi/eval && python lm_eval.py --ckpt {CK} --tokens {TOKENS} --out /kaggle/working/report/lm_eval.json
!cd /kaggle/working/repo/hindi/eval && python gen_eval.py --ckpt {CK} --tokens {TOKENS} --out-prefix /kaggle/working/report/gen
!cd /kaggle/working/repo/hindi/eval && python attn_analysis.py --ckpt {CK} --tokens {TOKENS} --out-dir /kaggle/working/report/attn --name pretrained